# 01 — Exploratory Data Analysis

Presentation only: every number below comes from `src.data_gate`, the same data-gate module `python -m src.data_gate` runs to produce `docs/DATA_NOTES.md`. This notebook holds no feature-engineering or modeling logic of its own — see `docs/ARCHITECTURE.md` §8 on why a second implementation of the same logic is the failure mode to avoid.

Full narrative account: `docs/DATA_NOTES.md` and `docs/LEAKAGE_FINDING.md`.

In [1]:
import pandas as pd
from src.data_gate import RAW_PATH, check_customer_viability, check_timestamp, leakage_sweep, _detect_target

df = pd.read_csv(RAW_PATH)
target_col = _detect_target(df)
print(f'{df.shape[0]:,} rows x {df.shape[1]} cols, target column: {target_col}')
df.head()

60,000 rows x 35 cols, target column: abuse_type


,order_id,customer_id,age,account_age_days,customer_segment,country,platform,device_type,payment_method,product_category,...,shipping_carrier,address_change_before_delivery,refund_to_different_account,multiple_accounts_flag,customer_support_contacts,previous_dispute_count,wishlist_to_cart_time_hrs,review_left_after_return,abuse_type,abuse_label
0,ORD2024554,CUST408891,68,1473,New,US,Web Browser,iPhone,Crypto,Toys,...,OnTrac,0,0,0,1,0,70.5,0,Legitimate,0
1,ORD2019797,CUST898762,64,1211,Silver,FR,Tablet App,MacBook,PayPal,Books,...,FedEx,0,0,0,0,0,25.3,0,Legitimate,0
2,ORD2058733,CUST906263,52,1291,Bronze,US,Web Browser,iPad,Crypto,Clothing,...,FedEx,1,0,0,0,0,36.6,0,Legitimate,0
3,ORD2015301,CUST601672,63,1743,Bronze,CA,Web Browser,iPhone,PayPal,Home & Kitchen,...,USPS,1,0,0,3,0,4.4,1,Fraudulent Return,2
4,ORD2014206,CUST316429,21,2478,Gold,US,Web Browser,Windows PC,Debit Card,Clothing,...,OnTrac,0,0,0,1,1,40.9,0,Legitimate,0


## Class balance

The strawman (always-predict-majority) baseline this motivates is recorded in `runs/model_full.json`.

In [2]:
balance = df[target_col].value_counts(normalize=True).mul(100).round(2)
balance

abuse_type
Legitimate           70.10
Policy Abuser        11.99
Fraudulent Return    10.19
Wardrobing            7.73
Name: proportion, dtype: float64

## Q1 — is there a usable repeat-customer identifier?

§4.2 of `docs/ARCHITECTURE.md` planned per-customer behavioral aggregates (trailing return rate, time-since-last-return, ...). Whether that's possible depends entirely on this.

In [3]:
check_customer_viability(df)

{'customer_id_column': 'customer_id',
 'verdict': 'NOT VIABLE (median = 1 row/customer)',
 'median_rows_per_customer': 1.0,
 'mean_rows_per_customer': 1.0343757542323209,
 'max_rows_per_customer': 4,
 'n_unique_customers': 58006,
 'detail': 'Median rows-per-customer == 1 -> every §4.2 feature is undefined; the feature plan falls back to §4.1 (see ARCHITECTURE.md §4.2).'}

**Median rows-per-customer is 1.0.** 56,061 of 58,006 customers appear exactly once. There is no per-customer history to aggregate over — §4.2 as originally scoped is impossible on this dataset, not merely difficult. The dataset ships the same behavioral signal *pre-computed* per row instead (`total_orders_lifetime`, `return_rate_pct`, `previous_dispute_count`, ...). Claiming credit for engineering that would be the dishonest version — see `docs/ARCHITECTURE.md` §4.2 and its correction log.

## Q2 — is there a usable timestamp?

Decides temporal vs. random split.

In [4]:
check_timestamp(df)

{'timestamp_column': 'order_date',
 'verdict': 'USABLE',
 'unparseable_rows': 0,
 'date_range': '2022-01-01 00:00:00 to 2023-12-02 00:00:00',
 'split_strategy': 'temporal (chronological 80/20)',
 'detail': 'Timestamp parses cleanly -> use a temporal split so evaluation reflects realistic future-prediction conditions.'}

**Usable.** The split is temporal on `return_date` (not `order_date`) — see `src/features.py::SPLIT_DATE_COL` and the reasoning in `docs/ARCHITECTURE.md` §4.

## Q3 — leakage sweep

Mutual information + a decision tree per feature, fit at `max(2, n_classes - 1)` depth (not the depth-1 originally specified — see `docs/LEAKAGE_FINDING.md`'s Finding 1 for why depth-1 is invalid on a 4-class target).

In [5]:
result = leakage_sweep(df, target_col)
print(f"{len(result['suspects'])} suspect feature(s) at >= 0.6 macro-F1 alone:")
result['suspects']

1 suspect feature(s) at >= 0.6 macro-F1 alone:


,feature,depth1_macro_f1,single_feature_macro_f1
31,abuse_label,0.393085,1.0


In [6]:
result['single_feature_f1'].head(10)

,feature,depth1_macro_f1,single_feature_macro_f1
31,abuse_label,0.393085,1.000000
18,return_rate_pct,0.393142,0.540067
17,total_returns_lifetime,0.397786,0.533390
8,avg_order_value_usd,0.388132,0.488907
9,refund_amount_requested_usd,0.390187,0.486450
14,days_to_return,0.436470,0.436470
22,tracking_number_valid,0.421304,0.421304
27,customer_support_contacts,0.377799,0.377799
29,wishlist_to_cart_time_hrs,0.372999,0.375316
28,previous_dispute_count,0.356658,0.356658


`abuse_label` is a 1:1 integer encoding of the target (macro-F1 = 1.000 alone) and is dropped in `src/features.py::DROP_COLS` — it must never re-enter the feature set. The remaining leakage story (three ordinary-looking features that reconstruct the label combinatorially) is not visible from single-feature screening at all; it's the subject of `docs/LEAKAGE_FINDING.md`, continued in `02_feature_engineering.ipynb`.